# Paper results — unit tests (first → fourth)

Builds every unit-test figure and table for the ShearNet paper from the benchmark
outputs of the four unit tests, and saves publication PDFs into `paper_plots/`
(created next to this notebook) plus paste-ready LaTeX rows as `.tex` files.

Inputs per tier (auto-discovered under each `research/unit_tests/<tier>/`):
- `benchmarking/m/metacal_results.fits`  (TAB_P / TAB_M)
- `benchmarking/psf_leakage/psf_leakage_results.fits`
- `timing/timing_results_{fair,realistic}.npz`

Tiers whose files are missing are skipped with a message, so the notebook can be
re-run as suite results accumulate. Ablation figures/tables are deliberately out
of scope (separate notebook once the unit tests are settled).

In [ ]:
import glob
import os
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
from astropy.table import Table

# ---------------- configuration ----------------
REPO_ROOT = Path.cwd().parent            # notebook lives in ShearNet/notebooks/
UNIT_ROOT = REPO_ROOT / "research" / "unit_tests"
OUT_DIR   = Path.cwd()

TIERS = ["first", "second", "third", "fourth"]
TIER_LABEL = {
    "first":  "I: Gaussian PSF, $g_1,g_2$",
    "second": "II: SuperBIT PSF, $g_1,g_2$",
    "third":  "III: + catalog HLR",
    "fourth": "IV: + catalog flux",
}

SHEAR_TRUE = 0.01      # eval.bias.shear_true
NJAC       = 20        # jackknife blocks
STAGE_IV   = 1e-3      # |m| requirement band

# method styling (consistent across every figure)
M_NG = dict(label="NGmix metacal", color="#3b7ea1", marker="o")
M_SN = dict(label="ShearNet",      color="#c0392b", marker="s")

plt.rcParams.update({
    "font.family": "serif", "mathtext.fontset": "dejavuserif",
    "axes.labelsize": 11, "axes.titlesize": 11,
    "legend.fontsize": 9, "xtick.labelsize": 10, "ytick.labelsize": 10,
    "figure.dpi": 110, "savefig.bbox": "tight",
})

def savefig(fig, name):
    p = OUT_DIR / name
    fig.savefig(p)
    print(f"saved -> {p.relative_to(Path.cwd())}")

def write_tex(name, text):
    p = OUT_DIR / name
    p.write_text(text)
    print(f"saved -> {p.relative_to(Path.cwd())}")

In [ ]:
# ---------------- discovery + loading ----------------
def _find(root, pattern):
    hits = sorted(glob.glob(str(root / "**" / pattern), recursive=True))
    return hits[0] if hits else None

def load_tier(tier):
    root = UNIT_ROOT / tier
    out = {"root": root}
    f = _find(root, "metacal_results*.fits")
    if f:
        with fits.open(f) as hdul:
            out["tab_p"] = Table(hdul["TAB_P"].data)
            out["tab_m"] = Table(hdul["TAB_M"].data)
        out["m_file"] = f
    f = _find(root, "psf_leakage_results*.fits")
    if f:
        out["lk"] = Table.read(f)
        out["lk_file"] = f
    for npz in sorted(glob.glob(str(root / "**" / "timing_results*.npz"), recursive=True)):
        mode = "realistic" if "realistic" in os.path.basename(npz) else "fair"
        out.setdefault("timing", {})[mode] = dict(np.load(npz))
    return out

RES = {t: load_tier(t) for t in TIERS}

print(f"{'tier':8s}  {'m fits':7s}  {'leakage':8s}  timing")
for t in TIERS:
    r = RES[t]
    print(f"{t:8s}  {'yes' if 'tab_p' in r else '--':7s}  "
          f"{'yes' if 'lk' in r else '--':8s}  "
          f"{', '.join(sorted(r.get('timing', {}))) or '--'}")

## Multiplicative & additive bias (`tab:bias` + bias ladder)

Response-corrected m and c per tier per method, mirroring the benchmark's
`jackknife_mc_v2` exactly: $\gamma_1 = (g_1^{+}-g_1^{-})/2$ per pair,
$m = \langle\gamma_1\rangle / \langle R_{11}\rangle / g_{\rm true} - 1$;
$c$ is the mean of the unsheared component $(g_2^{+}+g_2^{-})/2$. NGmix uses
`g_noshear`/`r11`; ShearNet uses `g_sn_noshear`/`r11_sn` (both the response- and
PSF-corrected columns the benchmark writes). Errors are the standard delete-one-block
jackknife with 20 blocks. Note the bias run applies shear to $g_1$ only, so this
measures $m_1$ and $c_2$; an independent $g_2$-sheared run would be needed for $m_2$.

In [ ]:
def col(tab, *names):
    for n in names:
        if n in tab.colnames:
            return np.asarray(tab[n])
    return None

def jackknife_mc(tab_p, tab_m, g_col, r11_col, shear_true=SHEAR_TRUE, njac=NJAC):
    """Mirror of research/shear_bias/m/helpers.jackknife_mc_v2 (m, c + errors)."""
    g_p, g_m = np.asarray(tab_p[g_col], float), np.asarray(tab_m[g_col], float)
    R_p, R_m = np.asarray(tab_p[r11_col], float), np.asarray(tab_m[r11_col], float)
    gamma1 = (g_p[:, 0] - g_m[:, 0]) / 2.0
    c_per  = (g_p[:, 1] + g_m[:, 1]) / 2.0
    R_pair = 0.5 * (R_p + R_m)
    chunks = np.array_split(np.arange(len(tab_p)), njac)
    m_j, c_j = [], []
    for ch in chunks:
        mask = np.ones(len(tab_p), bool); mask[ch] = False
        m_j.append(np.nanmean(gamma1[mask]) / np.nanmean(R_pair[mask]) / shear_true - 1)
        c_j.append(np.nanmean(c_per[mask]))
    m_j, c_j = np.array(m_j), np.array(c_j)
    m  = np.nanmean(gamma1) / np.nanmean(R_pair) / shear_true - 1
    c  = np.nanmean(c_per)
    m_err = np.sqrt((njac - 1) * np.mean((m_j - m_j.mean()) ** 2))
    c_err = np.sqrt((njac - 1) * np.mean((c_j - c_j.mean()) ** 2))
    return m, m_err, c, c_err

BIAS = {}   # BIAS[tier][method] = (m, m_err, c, c_err)
for t in TIERS:
    r = RES[t]
    if "tab_p" not in r:
        print(f"[{t}] no m fits -- skipped"); continue
    BIAS[t] = {}
    BIAS[t]["ngmix"] = jackknife_mc(r["tab_p"], r["tab_m"], "g_noshear", "r11")
    if "g_sn_noshear" in r["tab_p"].colnames and np.isfinite(
            np.asarray(r["tab_p"]["g_sn_noshear"], float)).any():
        BIAS[t]["shearnet"] = jackknife_mc(r["tab_p"], r["tab_m"], "g_sn_noshear", "r11_sn")
    for meth, v in BIAS[t].items():
        print(f"[{t}] {meth:9s} m1 = {v[0]*100:+.3f} +/- {v[1]*100:.3f} %   "
              f"c2 = {v[2]:+.2e} +/- {v[3]:.2e}")

In [ ]:
# ---- bias ladder figure + LaTeX table ----
if BIAS:
    fig, (ax0, ax1) = plt.subplots(2, 1, figsize=(6.0, 6.4), sharex=True)
    tiers = [t for t in TIERS if t in BIAS]
    x = np.arange(len(tiers))
    for meth, style, dx in [("ngmix", M_NG, -0.08), ("shearnet", M_SN, +0.08)]:
        vals = [(BIAS[t][meth] if meth in BIAS[t] else (np.nan,)*4) for t in tiers]
        m,  me = np.array([v[0] for v in vals]), np.array([v[1] for v in vals])
        c,  ce = np.array([v[2] for v in vals]), np.array([v[3] for v in vals])
        ax0.errorbar(x + dx, m, yerr=me, ls="", capsize=3, ms=5, **style)
        ax1.errorbar(x + dx, c, yerr=ce, ls="", capsize=3, ms=5, **style)
    ax0.axhspan(-STAGE_IV, STAGE_IV, color="0.85", zorder=0,
                label=r"Stage IV $|m|\lesssim 10^{-3}$")
    ax0.axhline(0, color="0.4", lw=0.8); ax1.axhline(0, color="0.4", lw=0.8)
    ax0.set_ylabel(r"$m_1$"); ax1.set_ylabel(r"$c_2$")
    ax1.set_xticks(x, [TIER_LABEL[t] for t in tiers])
    ax0.legend(); ax0.set_title("Shear bias across the unit-test ladder")
    fig.tight_layout(); savefig(fig, "unit_tests_bias_ladder.pdf"); plt.show()

    rows = []
    for t in tiers:
        for meth, name in [("ngmix", "NGmix metacalibration"), ("shearnet", r"\textsc{ShearNet}")]:
            if meth not in BIAS[t]: continue
            m, me, c, ce = BIAS[t][meth]
            rows.append(f"{t} & {name} & ${m*1e3:+.2f} \\pm {me*1e3:.2f}$ & "
                        f"${c*1e4:+.2f} \\pm {ce*1e4:.2f}$ \\\\")
    tex = ("% tab:bias -- m1 in units of 1e-3, c2 in units of 1e-4, jackknife errors\n"
           "% columns: tier & method & m1 [1e-3] & c2 [1e-4]\n" + "\n".join(rows) + "\n")
    write_tex("tab_bias.tex", tex)
    print(tex)

## PSF leakage (`fig:psf_leakage` + $\alpha$ ladder)

Recovered shear on unsheared galaxies vs the PSF ellipticity used to convolve
them. Convention established in the benchmark: NGmix uses the response- and
PSF-corrected `g`; ShearNet uses `g_sn_raw` (its physical, real-image
measurement). $\alpha_i$ is the least-squares slope of $g_i$ against
$g^{\rm PSF}_i$, with jackknife errors.

In [ ]:
def alpha_fit(gout, gpsf, njac=NJAC):
    ok = np.isfinite(gout) & np.isfinite(gpsf)
    gout, gpsf = gout[ok], gpsf[ok]
    def _slope(y, x):
        A = np.vstack([x, np.ones_like(x)]).T
        return np.linalg.lstsq(A, y, rcond=None)[0]
    a_full, b_full = _slope(gout, gpsf)
    chunks = np.array_split(np.arange(len(gout)), njac)
    a_j = []
    for ch in chunks:
        mask = np.ones(len(gout), bool); mask[ch] = False
        a_j.append(_slope(gout[mask], gpsf[mask])[0])
    a_j = np.array(a_j)
    a_err = np.sqrt((njac - 1) * np.mean((a_j - a_j.mean()) ** 2))
    return a_full, a_err, b_full

def binned_mean(x, y, nbins=12):
    ok = np.isfinite(x) & np.isfinite(y)
    x, y = x[ok], y[ok]
    edges = np.quantile(x, np.linspace(0, 1, nbins + 1))
    idx = np.digitize(x, edges[1:-1])
    bx, by, be = [], [], []
    for b in range(nbins):
        m = idx == b
        if m.sum() > 10:
            bx.append(x[m].mean()); by.append(y[m].mean())
            be.append(y[m].std() / np.sqrt(m.sum()))
    return map(np.array, (bx, by, be))

ALPHA = {}   # ALPHA[tier][method] = [(a1, a1e), (a2, a2e)]
for t in TIERS:
    r = RES[t]
    if "lk" not in r:
        print(f"[{t}] no leakage fits -- skipped"); continue
    lk = r["lk"]
    gpsf = np.asarray(lk["gpsf"], float)
    series = [("ngmix", np.asarray(lk["g"], float), M_NG)]
    if "g_sn_raw" in lk.colnames and np.isfinite(np.asarray(lk["g_sn_raw"], float)).any():
        series.append(("shearnet", np.asarray(lk["g_sn_raw"], float), M_SN))

    ALPHA[t] = {}
    fig, axes = plt.subplots(1, 2, figsize=(9.2, 3.8), sharey=True)
    for comp, ax in enumerate(axes):
        for meth, g, style in series:
            a, ae, b = alpha_fit(g[:, comp], gpsf[:, comp])
            if comp == 0: ALPHA[t][meth] = []
            ALPHA[t][meth].append((a, ae))
            bx, by, be = binned_mean(gpsf[:, comp], g[:, comp])
            ax.errorbar(bx, by, yerr=be, ls="", capsize=2, ms=4, **style)
            xx = np.linspace(np.nanmin(gpsf[:, comp]), np.nanmax(gpsf[:, comp]), 50)
            ax.plot(xx, a * xx + b, color=style["color"], lw=1.2,
                    label=rf"{style['label']}: $\alpha_{comp+1}={a:+.4f}\pm{ae:.4f}$")
        ax.axhline(0, color="0.4", lw=0.8)
        ax.set_xlabel(rf"$g^{{\rm PSF}}_{comp+1}$")
        if comp == 0: ax.set_ylabel(r"$\langle g_{\rm out}\rangle$")
        ax.legend(loc="best")
    fig.suptitle(f"PSF leakage -- unit test {t}", y=1.02)
    fig.tight_layout(); savefig(fig, f"psf_leakage_{t}.pdf"); plt.show()

# alpha ladder + LaTeX
if ALPHA:
    tiers = [t for t in TIERS if t in ALPHA]
    x = np.arange(len(tiers))
    fig, ax = plt.subplots(figsize=(6.0, 3.6))
    for meth, style, dx in [("ngmix", M_NG, -0.08), ("shearnet", M_SN, +0.08)]:
        a1  = np.array([ALPHA[t][meth][0][0] if meth in ALPHA[t] else np.nan for t in tiers])
        a1e = np.array([ALPHA[t][meth][0][1] if meth in ALPHA[t] else np.nan for t in tiers])
        ax.errorbar(x + dx, a1, yerr=a1e, ls="", capsize=3, ms=5, **style)
    ax.axhline(0, color="0.4", lw=0.8)
    ax.set_xticks(x, [TIER_LABEL[t] for t in tiers])
    ax.set_ylabel(r"$\alpha_1$"); ax.set_title("PSF leakage across the unit-test ladder")
    ax.legend(); fig.tight_layout()
    savefig(fig, "unit_tests_alpha_ladder.pdf"); plt.show()

    rows = []
    for t in tiers:
        for meth, name in [("ngmix", "NGmix metacalibration"), ("shearnet", r"\textsc{ShearNet}")]:
            if meth not in ALPHA[t]: continue
            (a1, a1e), (a2, a2e) = ALPHA[t][meth]
            rows.append(f"{t} & {name} & ${a1:+.4f} \\pm {a1e:.4f}$ & "
                        f"${a2:+.4f} \\pm {a2e:.4f}$ \\\\")
    tex = ("% fig:psf_leakage legend values / alpha table\n"
           "% columns: tier & method & alpha_1 & alpha_2\n" + "\n".join(rows) + "\n")
    write_tex("tab_alpha.tex", tex)
    print(tex)

## Bias vs S/N and size (`fig:snr_size`)

m recomputed in bins of galaxy S/N and of size ratio $T/T_{\rm PSF}$ (both
taken from the unsheared NGmix measurement of the + catalog, so the binning is
identical for the two methods). Jackknife errors within each bin.

In [ ]:
def m_in_bins(tab_p, tab_m, prop, edges, g_col, r11_col):
    g_p, g_m = np.asarray(tab_p[g_col], float), np.asarray(tab_m[g_col], float)
    R = 0.5 * (np.asarray(tab_p[r11_col], float) + np.asarray(tab_m[r11_col], float))
    gamma1 = (g_p[:, 0] - g_m[:, 0]) / 2.0
    idx = np.digitize(prop, edges[1:-1])
    bx, bm, be = [], [], []
    for b in range(len(edges) - 1):
        msk = (idx == b) & np.isfinite(gamma1) & np.isfinite(R) & np.isfinite(prop)
        if msk.sum() < 200: continue
        njac = min(NJAC, max(2, msk.sum() // 100))
        chunks = np.array_split(np.where(msk)[0], njac)
        m_j = []
        for ch in chunks:
            m2 = msk.copy(); m2[ch] = False
            m_j.append(np.nanmean(gamma1[m2]) / np.nanmean(R[m2]) / SHEAR_TRUE - 1)
        m_j = np.array(m_j)
        bx.append(np.nanmedian(prop[msk]))
        bm.append(np.nanmean(gamma1[msk]) / np.nanmean(R[msk]) / SHEAR_TRUE - 1)
        be.append(np.sqrt((njac - 1) * np.mean((m_j - m_j.mean()) ** 2)))
    return map(np.array, (bx, bm, be))

for t in TIERS:
    r = RES[t]
    if "tab_p" not in r: continue
    tp, tm = r["tab_p"], r["tab_m"]
    s2n  = col(tp, "s2n_noshear", "s2n")
    Tgal = col(tp, "T_noshear", "T")
    Tpsf = col(tp, "Tpsf_noshear", "Tpsf")
    panels = []
    if s2n is not None:
        panels.append(("S/N", np.asarray(s2n, float), "log"))
    if Tgal is not None and Tpsf is not None:
        panels.append((r"$T/T_{\rm PSF}$",
                       np.asarray(Tgal, float) / np.asarray(Tpsf, float), "linear"))
    if not panels:
        print(f"[{t}] no s2n/T columns -- skipped"); continue

    fig, axes = plt.subplots(1, len(panels), figsize=(4.8 * len(panels), 3.8), sharey=True)
    axes = np.atleast_1d(axes)
    for ax, (xlabel, prop, xscale) in zip(axes, panels):
        lo, hi = np.nanquantile(prop, [0.01, 0.99])
        edges = (np.geomspace(max(lo, 1e-3), hi, 9) if xscale == "log"
                 else np.linspace(lo, hi, 9))
        for meth, gcol, rcol, style in [("ngmix", "g_noshear", "r11", M_NG),
                                        ("shearnet", "g_sn_noshear", "r11_sn", M_SN)]:
            if gcol not in tp.colnames: continue
            if not np.isfinite(np.asarray(tp[gcol], float)).any(): continue
            bx, bm, be = m_in_bins(tp, tm, prop, edges, gcol, rcol)
            ax.errorbar(bx, bm, yerr=be, ls="-", lw=0.8, capsize=2, ms=4, **style)
        ax.axhspan(-STAGE_IV, STAGE_IV, color="0.85", zorder=0)
        ax.axhline(0, color="0.4", lw=0.8)
        ax.set_xlabel(xlabel); ax.set_xscale(xscale)
    axes[0].set_ylabel(r"$m_1$"); axes[0].legend()
    fig.suptitle(f"Bias vs S/N and size -- unit test {t}", y=1.02)
    fig.tight_layout(); savefig(fig, f"m_vs_snr_size_{t}.pdf"); plt.show()

## Throughput (`tab:timing`) and per-galaxy accuracy (`tab:baseline`)

Timing comes straight from the benchmark npz (throughput in gal/s and the
speedup factor, for whichever of the fair/realistic modes were run). The
accuracy table reports per-galaxy MSE against the stored truths where they
exist: size (ShearNet `g_sn_sigma` vs `gal_hlr_th`; tiers III-IV) and flux
(`g_sn_flux` vs `gal_flux_th`; tier IV), plus the per-pair shear scatter
$\sigma(\gamma_1)$ for both methods (shape noise is common mode, so the
*difference* between methods is the meaningful part). The paper's MSE$(g_1,g_2)$
column against per-galaxy truth requires the standard eval outputs and the
concat-fusion row requires that ablation model -- both left as [XX] here.

In [ ]:
# ---- timing table ----
rows = []
for t in TIERS:
    for mode, d in sorted(RES[t].get("timing", {}).items()):
        ng = float(d.get("ngmix_throughput", np.nan))
        sn = float(d.get("shearnet_throughput", np.nan))
        sp = float(d.get("speedup_throughput", np.nan))
        ncpu = int(d["nproc"]) if "nproc" in d else "[NN]"
        rows.append((t, mode, ng, sn, sp, ncpu))
        print(f"[{t}/{mode}] ngmix {ng:9.1f} gal/s ({ncpu} cores)   "
              f"shearnet {sn:9.1f} gal/s (1 GPU)   speedup {sp:6.1f}x")
if rows:
    tex_rows = [f"{t} ({mode}) & NGmix metacalibration & {ncpu} CPU cores & {ng:.0f} \\\\\n"
                f"{t} ({mode}) & \\textsc{{ShearNet}} & 1 GPU & {sn:.0f} \\\\\n"
                f"{t} ({mode}) & Speedup & & ${{\\sim}}{sp:.0f}\\times$ \\\\"
                for t, mode, ng, sn, sp, ncpu in rows]
    tex = ("% tab:timing -- throughput (gal/s)\n" + "\n".join(tex_rows) + "\n")
    write_tex("tab_timing.tex", tex)
else:
    print("no timing npz found in any tier")

In [ ]:
# ---- per-galaxy accuracy table ----
rows = []
for t in TIERS:
    r = RES[t]
    if "tab_p" not in r: continue
    tp, tm = r["tab_p"], r["tab_m"]
    g_p, g_m = np.asarray(tp["g_noshear"], float), np.asarray(tm["g_noshear"], float)
    gam_ng = (g_p[:, 0] - g_m[:, 0]) / 2.0
    line = {"tier": t, "sig_g_ng": np.nanstd(gam_ng)}
    if "g_sn_noshear" in tp.colnames and np.isfinite(np.asarray(tp["g_sn_noshear"], float)).any():
        gs_p = np.asarray(tp["g_sn_noshear"], float)
        gs_m = np.asarray(tm["g_sn_noshear"], float)
        line["sig_g_sn"] = np.nanstd((gs_p[:, 0] - gs_m[:, 0]) / 2.0)
    hlr_th  = col(tp, "gal_hlr_th");  sn_sig = col(tp, "g_sn_sigma")
    flux_th = col(tp, "gal_flux_th"); sn_flx = col(tp, "g_sn_flux")
    if hlr_th is not None and sn_sig is not None and np.isfinite(sn_sig).any():
        line["mse_sigma"] = np.nanmean((np.asarray(sn_sig) - np.asarray(hlr_th)) ** 2)
    if flux_th is not None and sn_flx is not None and np.isfinite(np.asarray(sn_flx)).any():
        fth = np.asarray(flux_th, float)
        line["mse_flux_rel"] = np.nanmean(((np.asarray(sn_flx) - fth) / fth) ** 2)
    rows.append(line)

if rows:
    hdr = f"{'tier':8s} {'sig(g1) ng':>11s} {'sig(g1) sn':>11s} {'MSE(hlr) sn':>12s} {'relMSE(F) sn':>13s}"
    print(hdr); print("-" * len(hdr))
    tex_rows = []
    for l in rows:
        fmt  = lambda k, s, w: (s.format(l[k]) if k in l else "--".rjust(w))
        fmtx = lambda k, s: (s.format(l[k]) if k in l else "--")
        print(f"{l['tier']:8s} {fmt('sig_g_ng', '{:11.4f}', 11)} {fmt('sig_g_sn', '{:11.4f}', 11)} "
              f"{fmt('mse_sigma', '{:12.3e}', 12)} {fmt('mse_flux_rel', '{:13.3e}', 13)}")
        tex_rows.append(f"{l['tier']} & {fmtx('sig_g_ng', '{:.4f}')} & {fmtx('sig_g_sn', '{:.4f}')} & "
                        f"{fmtx('mse_sigma', '{:.3e}')} & {fmtx('mse_flux_rel', '{:.3e}')} \\\\")
    tex = ("% tab:baseline (partial) -- per-pair shear scatter + MSE vs stored truths\n"
           "% columns: tier & sigma(gamma1) ngmix & sigma(gamma1) shearnet & "
           "MSE(hlr) shearnet & relative MSE(flux) shearnet\n" + "\n".join(tex_rows) + "\n")
    write_tex("tab_baseline.tex", tex)

## Scope notes

Produced here, from the three benchmark outputs per tier: the bias ladder
(`unit_tests_bias_ladder.pdf`), per-tier leakage figures + $\alpha$ ladder,
per-tier $m$ vs S/N & size figures, and the bias / alpha / timing / accuracy
tables as paste-ready `.tex` rows.

**Not derivable from these files** (tracked separately): the $\rho$-statistics
figure (TreeCorr on reserved-star PSF residuals), `psf_properties_superbit.pdf`
(PSFEx spatial maps), the MSE$(g_1,g_2)$ column of `tab:baseline` (needs the
standard eval outputs) and its concat-fusion row, and all appendix
ablation/hyperparameter tables (separate notebook once the unit tests are
settled). Re-running this notebook refreshes every PDF and `.tex` in place.